# Advanced Problems: Named Tuple Docstrings and Default Values

This notebook contains advanced practice problems with full solutions for Python `namedtuple` docstrings and default values.

## Topics covered

- Setting class-level docstrings on generated named tuple classes.
- Setting field-level docstrings on named tuple properties.
- Creating prototype instances with default values.
- Using `_replace` with prototype records.
- Setting constructor defaults through `__new__.__defaults__`.
- Understanding right-aligned default values.
- Avoiding mutable default values.
- Building reusable utilities for documented and defaulted named tuples.


In [1]:
from collections import namedtuple
import inspect

## Problem 1 — Document a named tuple class and its fields

Create a `Point2D` named tuple with fields `x` and `y`.

Then add:

- A class docstring: `'Represents a point in a 2D Cartesian coordinate system.'`
- A field docstring for `x`: `'Horizontal coordinate.'`
- A field docstring for `y`: `'Vertical coordinate.'`

Finally, verify the docstrings programmatically without calling `help()`.

### Solution

In [2]:
Point2D = namedtuple('Point2D', 'x y')

Point2D.__doc__ = 'Represents a point in a 2D Cartesian coordinate system.'
Point2D.x.__doc__ = 'Horizontal coordinate.'
Point2D.y.__doc__ = 'Vertical coordinate.'

print(Point2D.__doc__)
print(Point2D.x.__doc__)
print(Point2D.y.__doc__)

assert Point2D.__doc__ == 'Represents a point in a 2D Cartesian coordinate system.'
assert Point2D.x.__doc__ == 'Horizontal coordinate.'
assert Point2D.y.__doc__ == 'Vertical coordinate.'

Represents a point in a 2D Cartesian coordinate system.
Horizontal coordinate.
Vertical coordinate.


### Best-practice note

Changing `__doc__` does not change the behavior of the named tuple. It improves introspection, documentation, and interactive help output.

## Problem 2 — Build a reusable documentation helper

Write a function:

```python
document_namedtuple(cls, class_doc, field_docs)
```

It should:

1. Set the class docstring.
2. Set field docstrings using a dictionary.
3. Raise `ValueError` if `field_docs` contains a field that does not exist.
4. Return the same class object, so the function can be used fluently.


In [3]:
Vector = namedtuple('Vector', 'x1 y1 x2 y2 origin_x origin_y')

### Solution

In [4]:
def document_namedtuple(cls, class_doc, field_docs):
    valid_fields = set(cls._fields)
    unknown_fields = set(field_docs) - valid_fields

    if unknown_fields:
        raise ValueError(f'Unknown field(s): {sorted(unknown_fields)}')

    cls.__doc__ = class_doc

    for field_name, doc in field_docs.items():
        getattr(cls, field_name).__doc__ = doc

    return cls


Vector = document_namedtuple(
    Vector,
    'Represents a directed line segment with an optional origin.',
    {
        'x1': 'Starting x-coordinate.',
        'y1': 'Starting y-coordinate.',
        'x2': 'Ending x-coordinate.',
        'y2': 'Ending y-coordinate.',
        'origin_x': 'Origin x-coordinate.',
        'origin_y': 'Origin y-coordinate.'
    }
)

assert Vector.__doc__ == 'Represents a directed line segment with an optional origin.'
assert Vector.x1.__doc__ == 'Starting x-coordinate.'
assert Vector.origin_y.__doc__ == 'Origin y-coordinate.'

print(Vector.__doc__)
print(Vector.x1.__doc__)
print(Vector.origin_y.__doc__)

Represents a directed line segment with an optional origin.
Starting x-coordinate.
Origin y-coordinate.


In [5]:
try:
    document_namedtuple(Vector, 'Bad docs', {'missing': 'This field does not exist.'})
except ValueError as ex:
    print(type(ex).__name__, ex)

ValueError Unknown field(s): ['missing']


### Why this is useful

For larger named tuples, documenting each property manually is repetitive. A helper makes the process consistent and validates the documentation keys.

## Problem 3 — Create defaults using a prototype instance

Create a `Rectangle` named tuple with fields:

`x y width height fill border`

Use a prototype named `default_rectangle` with these defaults:

- `x = 0`
- `y = 0`
- `width = 1`
- `height = 1`
- `fill = 'white'`
- `border = 'black'`

Then create two rectangles using `_replace`:

1. A large red rectangle.
2. A small blue rectangle with no custom border.


### Solution

In [6]:
Rectangle = namedtuple('Rectangle', 'x y width height fill border')

default_rectangle = Rectangle(
    x=0,
    y=0,
    width=1,
    height=1,
    fill='white',
    border='black'
)

large_red = default_rectangle._replace(
    width=100,
    height=50,
    fill='red'
)

small_blue = default_rectangle._replace(
    x=10,
    y=20,
    fill='blue'
)

print(default_rectangle)
print(large_red)
print(small_blue)

assert large_red == Rectangle(0, 0, 100, 50, 'red', 'black')
assert small_blue == Rectangle(10, 20, 1, 1, 'blue', 'black')
assert default_rectangle.fill == 'white'

Rectangle(x=0, y=0, width=1, height=1, fill='white', border='black')
Rectangle(x=0, y=0, width=100, height=50, fill='red', border='black')
Rectangle(x=10, y=20, width=1, height=1, fill='blue', border='black')


### Best-practice note

A prototype is useful when you want multiple sets of defaults. For example, you could have `default_button_rectangle`, `default_card_rectangle`, and `default_modal_rectangle`.

## Problem 4 — Multiple prototypes for the same named tuple

Using the same `Rectangle` type, create two prototypes:

- `light_theme_rectangle`: white fill, black border.
- `dark_theme_rectangle`: black fill, white border.

Write a function:

```python
make_rectangle(theme, **overrides)
```

It should use the correct prototype and return a rectangle with overrides applied.

Reject unknown themes.

### Solution

In [7]:
light_theme_rectangle = Rectangle(0, 0, 1, 1, 'white', 'black')
dark_theme_rectangle = Rectangle(0, 0, 1, 1, 'black', 'white')

PROTOTYPES = {
    'light': light_theme_rectangle,
    'dark': dark_theme_rectangle
}

def make_rectangle(theme, **overrides):
    try:
        prototype = PROTOTYPES[theme]
    except KeyError:
        raise ValueError(f'Unknown theme: {theme!r}') from None

    return prototype._replace(**overrides)


light_banner = make_rectangle('light', width=300, height=80)
dark_card = make_rectangle('dark', x=20, y=30, width=120, height=90)

print(light_banner)
print(dark_card)

assert light_banner.fill == 'white'
assert light_banner.border == 'black'
assert dark_card.fill == 'black'
assert dark_card.border == 'white'

Rectangle(x=0, y=0, width=300, height=80, fill='white', border='black')
Rectangle(x=20, y=30, width=120, height=90, fill='black', border='white')


In [8]:
try:
    make_rectangle('solarized', width=10)
except ValueError as ex:
    print(type(ex).__name__, ex)

ValueError Unknown theme: 'solarized'


### Design insight

Prototype-based defaults are especially useful when the defaults depend on a category, mode, theme, environment, or configuration.

## Problem 5 — Set constructor defaults using `__new__.__defaults__`

Create a `Vector` named tuple with fields:

`x1 y1 x2 y2 origin_x origin_y`

Set constructor defaults so that only `origin_x` and `origin_y` default to `0`.

Then verify that:

1. `Vector(1, 2, 3, 4)` works.
2. `Vector(1, 2, 3, 4, 10, 20)` still works.
3. `Vector()` fails because the first four fields are still required.

### Solution

In [9]:
Vector = namedtuple('Vector', 'x1 y1 x2 y2 origin_x origin_y')
Vector.__new__.__defaults__ = (0, 0)

v1 = Vector(1, 2, 3, 4)
v2 = Vector(1, 2, 3, 4, 10, 20)

print(v1)
print(v2)

assert v1 == Vector(1, 2, 3, 4, 0, 0)
assert v2 == Vector(1, 2, 3, 4, 10, 20)

try:
    Vector()
except TypeError as ex:
    print(type(ex).__name__, ex)

Vector(x1=1, y1=2, x2=3, y2=4, origin_x=0, origin_y=0)
Vector(x1=1, y1=2, x2=3, y2=4, origin_x=10, origin_y=20)
TypeError Vector.__new__() missing 4 required positional arguments: 'x1', 'y1', 'x2', and 'y2'


### Key rule

Defaults are right-aligned. If a named tuple has six fields and you set two defaults, those defaults apply to the last two fields.

## Problem 6 — Explain and test right-aligned defaults

Create a `Config` named tuple with fields:

`host port protocol timeout retries`

Set `__new__.__defaults__` to:

```python
('https', 30, 3)
```

Then determine which fields receive defaults and prove it with assertions.

### Solution

In [10]:
Config = namedtuple('Config', 'host port protocol timeout retries')
Config.__new__.__defaults__ = ('https', 30, 3)

cfg = Config('example.com', 443)

print(cfg)

assert cfg.host == 'example.com'
assert cfg.port == 443
assert cfg.protocol == 'https'
assert cfg.timeout == 30
assert cfg.retries == 3

print('Defaults apply to:', Config._fields[-len(Config.__new__.__defaults__):])

Config(host='example.com', port=443, protocol='https', timeout=30, retries=3)
Defaults apply to: ('protocol', 'timeout', 'retries')


### Explanation

The tuple `('https', 30, 3)` has three values, so it applies to the last three fields: `protocol`, `timeout`, and `retries`.

## Problem 7 — Give all fields the same default value

Create a `Pixel` named tuple with fields:

`red green blue alpha`

Set every field to default to `0` using `len(Pixel._fields)`.

Then create:

- `Pixel()`
- `Pixel(red=255)`
- `Pixel(red=255, green=128, blue=64, alpha=255)`

### Solution

In [11]:
Pixel = namedtuple('Pixel', 'red green blue alpha')
Pixel.__new__.__defaults__ = (0,) * len(Pixel._fields)

transparent_black = Pixel()
red_only = Pixel(red=255)
opaque_orange = Pixel(red=255, green=128, blue=64, alpha=255)

print(transparent_black)
print(red_only)
print(opaque_orange)

assert transparent_black == Pixel(0, 0, 0, 0)
assert red_only == Pixel(255, 0, 0, 0)
assert opaque_orange == Pixel(255, 128, 64, 255)

Pixel(red=0, green=0, blue=0, alpha=0)
Pixel(red=255, green=0, blue=0, alpha=0)
Pixel(red=255, green=128, blue=64, alpha=255)


### Practical warning

This pattern is safe for immutable defaults like numbers, strings, booleans, and `None`. Be careful with mutable defaults such as lists and dictionaries.

## Problem 8 — Demonstrate the mutable default pitfall

Create a `Task` named tuple with fields:

`title tags`

Set the default value for `tags` to an empty list using `__new__.__defaults__`.

Then show why this is dangerous by creating two tasks and modifying the tags list of one of them.

Finally, show a safer approach using `None` as the default and a factory function.

### Solution

In [12]:
Task = namedtuple('Task', 'title tags')
Task.__new__.__defaults__ = ([],)

task1 = Task('Write docs')
task2 = Task('Review PR')

task1.tags.append('urgent')

print(task1)
print(task2)
print('Same tags object?', task1.tags is task2.tags)

assert task1.tags is task2.tags

Task(title='Write docs', tags=['urgent'])
Task(title='Review PR', tags=['urgent'])
Same tags object? True


The two tasks share the same default list object. That is usually a bug.

A safer approach is to use `None` as the default and create a new list inside a factory function.

In [13]:
SafeTask = namedtuple('SafeTask', 'title tags')
SafeTask.__new__.__defaults__ = (None,)

def make_task(title, tags=None):
    if tags is None:
        tags = []
    else:
        tags = list(tags)
    return SafeTask(title, tags)

safe1 = make_task('Write docs')
safe2 = make_task('Review PR')

safe1.tags.append('urgent')

print(safe1)
print(safe2)
print('Same tags object?', safe1.tags is safe2.tags)

assert safe1.tags == ['urgent']
assert safe2.tags == []
assert safe1.tags is not safe2.tags

SafeTask(title='Write docs', tags=['urgent'])
SafeTask(title='Review PR', tags=[])
Same tags object? False


### Advanced takeaway

A named tuple instance is immutable in the sense that its fields cannot be reassigned, but the object stored inside a field may still be mutable.

## Problem 9 — Create a documented named tuple with defaults in one utility

Write a function:

```python
make_documented_namedtuple(type_name, fields, class_doc, field_docs, defaults=())
```

It should:

1. Create a named tuple class.
2. Set the class docstring.
3. Set field docstrings.
4. Set constructor defaults through `__new__.__defaults__`.
5. Validate that field documentation only refers to real fields.

Use it to create an `ApiRequest` type with fields:

`method url timeout retries headers`

Defaults should be:

- `timeout = 30`
- `retries = 3`
- `headers = None`


### Solution

In [14]:
def make_documented_namedtuple(type_name, fields, class_doc, field_docs, defaults=()):
    cls = namedtuple(type_name, fields)

    unknown_fields = set(field_docs) - set(cls._fields)
    if unknown_fields:
        raise ValueError(f'Unknown documented field(s): {sorted(unknown_fields)}')

    cls.__doc__ = class_doc

    for field_name, field_doc in field_docs.items():
        getattr(cls, field_name).__doc__ = field_doc

    cls.__new__.__defaults__ = tuple(defaults)

    return cls


ApiRequest = make_documented_namedtuple(
    'ApiRequest',
    'method url timeout retries headers',
    'Represents an HTTP API request configuration.',
    {
        'method': 'HTTP method such as GET or POST.',
        'url': 'Target request URL.',
        'timeout': 'Timeout in seconds.',
        'retries': 'Number of retry attempts.',
        'headers': 'Optional HTTP headers dictionary.'
    },
    defaults=(30, 3, None)
)

req = ApiRequest('GET', 'https://example.com')

print(req)
print(ApiRequest.__doc__)
print(ApiRequest.timeout.__doc__)

assert req.method == 'GET'
assert req.url == 'https://example.com'
assert req.timeout == 30
assert req.retries == 3
assert req.headers is None

ApiRequest(method='GET', url='https://example.com', timeout=30, retries=3, headers=None)
Represents an HTTP API request configuration.
Timeout in seconds.


### Best-practice note

This pattern is good for teaching and experimentation. In production code, consider whether `typing.NamedTuple`, `dataclasses.dataclass(frozen=True)`, or a validation library would be more appropriate.

## Problem 10 — Introspection report for a named tuple

Write a function `describe_namedtuple(cls)` that returns a dictionary containing:

- `type_name`
- `class_doc`
- `fields`
- `field_docs`
- `defaults`
- `required_fields`
- `defaulted_fields`

Use it on the `ApiRequest` type from the previous problem.

### Solution

In [15]:
def describe_namedtuple(cls):
    fields = cls._fields
    defaults = cls.__new__.__defaults__ or ()

    if defaults:
        defaulted_fields = fields[-len(defaults):]
        required_fields = fields[:-len(defaults)]
    else:
        defaulted_fields = ()
        required_fields = fields

    return {
        'type_name': cls.__name__,
        'class_doc': cls.__doc__,
        'fields': fields,
        'field_docs': {
            field: getattr(cls, field).__doc__
            for field in fields
        },
        'defaults': defaults,
        'required_fields': required_fields,
        'defaulted_fields': defaulted_fields
    }


report = describe_namedtuple(ApiRequest)
report

{'type_name': 'ApiRequest',
 'class_doc': 'Represents an HTTP API request configuration.',
 'fields': ('method', 'url', 'timeout', 'retries', 'headers'),
 'field_docs': {'method': 'HTTP method such as GET or POST.',
  'url': 'Target request URL.',
  'timeout': 'Timeout in seconds.',
  'retries': 'Number of retry attempts.',
  'headers': 'Optional HTTP headers dictionary.'},
 'defaults': (30, 3, None),
 'required_fields': ('method', 'url'),
 'defaulted_fields': ('timeout', 'retries', 'headers')}

In [16]:
assert report['type_name'] == 'ApiRequest'
assert report['fields'] == ('method', 'url', 'timeout', 'retries', 'headers')
assert report['defaults'] == (30, 3, None)
assert report['required_fields'] == ('method', 'url')
assert report['defaulted_fields'] == ('timeout', 'retries', 'headers')

## Final summary

In this notebook, you practiced advanced patterns for documenting and defaulting named tuples:

- Class docstrings can be set through `ClassName.__doc__`.
- Field docstrings can be set through `ClassName.field.__doc__`.
- Prototype objects are useful when different default profiles are needed.
- `_replace` creates modified copies from immutable named tuple instances.
- `__new__.__defaults__` sets constructor defaults.
- Constructor defaults are right-aligned with fields.
- Avoid mutable default values unless you fully understand the shared-object behavior.
- Utility functions can make named tuple documentation and default management safer and more consistent.
